# Retrieve & Re-Ranking LAB

Podstawą wszystkich rozwiązań typu RAG (ang. Retrieval Augmented Generation) jest komponent wyszukiwania. Wyszukiwanie polega na identyfikacji odpowiednich dokumentów, w naszym RAGOwym przypadku takich dokumentów, które potencjalnie zawierają informacje kluczowe do odpowiedzenia na postawione pytanie. Definicja dokumentu jest też rzeczą płynną, raz to może być paragraf, a raz strona, czy rozdział, zależy to od zadania, oraz modeli jakie używamy.

Na wykładzie starałem się wyraźnie zwrócić uwagę na to, że wyszukiwanie może być leksykalne (BM25) lub semantyczne (najczęściej oparte na głębokich sieciach neuronowych, używanych jako modeli embedderów tekstów lub ocaniejących dopasowanie pary tekst-pytanie).
Podczas wykładu dokładnie omówiliśmy dwie kluczowe architektury głębokich sieci Transformer stosowanych do wyszukiwania informacji - bi-encoder i cross-encoder.

Przykładem bi-encodera jest `SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')`


```
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("multi-qa-mpnet-base-dot-v1")

docs = [
    "My first paragraph. That contains information",
    "Python is a programming language.",
]
document_embeddings = model.encode(docs)

query = "What is Python?"
query_embedding = model.encode(query)

similarity_scores = embedder.similarity(query_embedding, document_embeddings)

```


Przykładem CrossEncoder (`CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')`), który w odróżnieniu od bi-encodera ocenia odpowiedniość dokumentu do zapytania, zwracając dla podanej pary tekstów ich wskaźnik dopasowania (prawdopodobieństwo dopasowania).

```
from sentence_transformers import CrossEncoder
import torch

# Load https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", activation_fn=torch.nn.Sigmoid())
scores = model.predict([
    ("How many people live in Berlin?", "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers."),
    ("How many people live in Berlin?", "Berlin is well known for its museums."),
])
```

Do pracy wymagamy zainstalowania trzech pakietów - `sentence-transformers` oraz `rank_bm25` oraz `datasets`


In [1]:
!pip install -U sentence-transformers rank_bm25 datasets

Zanim przejdziemy do zdefiniowania zadania przygotujemy sobie kilka przykładów użycia różnych metod i modeli, abyście mieli podstawy do realizacji swojego wyzwania.


Przykładowe użycie leksykalnego wyszukiwania opartego na BM25

In [2]:
from rank_bm25 import BM25Okapi

# Korpus z przykładowymi tekstami
corpus = [
    "Trzeba jeść dużo warzyw, owoców i uprawiać sport.",
    "Trzeba pić alkohol, imprezować i palić.",
    "Kampania polityczna bardzo napawa optymizmem, same mądre głowy chcą rządzić Polakami.",
    "Historia Polski to wiele wieków walki o niepodległość.",
    "Przemysł motoryzacyjny od lat walczy z rosnącymi kosztami produkcji.",
]

tokenized_corpus = [doc.lower().split(" ") for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

## KIEDY ZAPYTANIE MA LEKSYKALNE PRZECIĘCIE z KORPUSEM

query_0 = "O czym jest historia Polski?"
print(query_0)

tokenized_query_0 = query_0.lower().split(" ")

#policz dopasowanie leksykalne zapytania do każdego dokumentu w korpusie
doc_scores_0 = bm25.get_scores(tokenized_query_0)
print(doc_scores_0)

#znajdz top N najbliższych dokumentów do zadanego zapytania
print(bm25.get_top_n(tokenized_query_0, corpus, n=1))

## KIEDY ZAPYTANIE NIE MA LEKSYKALNEGO PRZECIĘCIE z KORPUSEM

query_1 = "Jak długo żyć?"
print(query_1)

tokenized_query_1 = query_1.lower().split(" ")

#policz dopasowanie leksykalne zapytania do każdego dokumentu w korpusie
doc_scores_1 = bm25.get_scores(tokenized_query_1)
print(doc_scores_1)

#znajdz top N najbliższych dokumentów do zadanego zapytania
print(bm25.get_top_n(tokenized_query_1, corpus, n=1))



O czym jest historia Polski?
[0.         0.         0.         2.24533898 0.        ]
['Historia Polski to wiele wieków walki o niepodległość.']
Jak długo żyć?
[0. 0. 0. 0. 0.]
['Przemysł motoryzacyjny od lat walczy z rosnącymi kosztami produkcji.']


Przykładowe użycie semantycznego wyszukiwania bazującego na rozwiazaniach typu bi-encoder dla j.polskiego.

In [3]:
import torch

from sentence_transformers import SentenceTransformer


embedder = SentenceTransformer("sdadas/mmlw-retrieval-roberta-large")

query_prefix = "zapytanie: "
answer_prefix = ""

# Korpus z przykładowymi tekstami
corpus = [
    answer_prefix + "Trzeba jeść dużo warzyw, owoców i uprawiać sport.",
    answer_prefix + "Trzeba pić alkohol, imprezować i palić.",
    answer_prefix + "Kampania polityczna bardzo napawa optymizmem, same mądre głowy chcą rządzić Polakami.",
    answer_prefix + "Historia Polski to wiele wieków walki o niepodległość.",
    answer_prefix + "Przemysł motoryzacyjny od lat walczy z rosnącymi kosztami produkcji.",
]



# Use "convert_to_tensor=True" to keep the tensors on GPU (if available)
corpus_embeddings = embedder.encode(corpus, convert_to_tensor=True)

# Zapytania
queries = [
    query_prefix + "Jak dożyć 100 lat?",
    query_prefix + "Z czym walczy sektor automoto?"
]

# Znajdz dwa najbliższe dokumenty z korpusu dla zadanego zapytania używajać podobieństwa opartego na mierze cosinus
top_k = min(2, len(corpus))
for query in queries:
    query_embedding = embedder.encode(query, convert_to_tensor=True)

    # We use cosine-similarity and torch.topk to find the highest 2 scores
    similarity_scores = embedder.similarity(query_embedding, corpus_embeddings)[0]
    scores, indices = torch.topk(similarity_scores, k=top_k)

    print("\nZapytanie:", query)
    print("Top 2 najbliższych dokumentów:")

    for score, idx in zip(scores, indices):
        print(corpus[idx], f"(Score: {score:.4f})")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Zapytanie: zapytanie: Jak dożyć 100 lat?
Top 2 najbliższych dokumentów:
Trzeba jeść dużo warzyw, owoców i uprawiać sport. (Score: 0.7192)
Trzeba pić alkohol, imprezować i palić. (Score: 0.6929)

Zapytanie: zapytanie: Z czym walczy sektor automoto?
Top 2 najbliższych dokumentów:
Przemysł motoryzacyjny od lat walczy z rosnącymi kosztami produkcji. (Score: 0.7715)
Historia Polski to wiele wieków walki o niepodległość. (Score: 0.6724)


Przykładowe użycie semantycznego wyszukiwania w zakresie komponentu re-rankera opartego na polskich cross-encoderach.

In [4]:
from sentence_transformers import CrossEncoder
import torch.nn

query = "Jak dożyć spokojnej starości w dobrej kondycji?"
answers = [
    "Trzeba zdrowo się odżywiać, oraz unikać alkoholu i papierosów",
    "Trzeba pić alkohol, imprezować i chwytać dzień",
    "Życie to tylko ciąg wspomnień i obrazy z przeszłości."
]

model = CrossEncoder(
    "sdadas/polish-reranker-large-ranknet",
    device="cuda" if torch.cuda.is_available() else "cpu"
)
pairs = [[query, answer] for answer in answers]
results = model.predict(pairs)
print(results.tolist())


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[0.8108768463134766, 0.3784073293209076, 0.0719650462269783]


ZADANIE #1 PREPROCESSING - przetwórz korpus polskiej wikipedii do postaci kolekcji tekstów

In [5]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [6]:
import json
import gzip
import os
import torch

if not torch.cuda.is_available():
    print("Warning: No GPU found. Please add GPU to your notebook")

# Użyj dumpa polskiej Wikipedii.

wikipedia_hf_path = 'chrisociepa/wikipedia-pl-20230401'

from datasets import load_dataset
from torch.utils.data import random_split
dataset = load_dataset(wikipedia_hf_path, split="train", streaming=True)
dataset = dataset.shuffle(seed=42)

##Napisz funkcję, która zbuduje min kilka set tys zbiór dokumentów zwanych pasażami
##Używając tej funkcji wypełnij zmienną passages, sprawdz i usun duplikaty
def create_passages(n_passages, dataset):
    def get_n_words(n_words, text):
        return " ".join(text.split(" ")[:n_words])
    passages = set()
    data = iter(dataset)
    for _ in range(n_passages):
        passages.add(get_n_words(64, next(data)['text']))
    return list(passages)
passages = create_passages(500000, dataset)

print("Passages:", len(passages))
print(passages[0])

Passages: 499874
Bruno Ochman (ur. 7 września 1929, zm. 3 listopada 1990) – kanadyjski zapaśnik walczący w stylu wolnym. Olimpijczyk z Melbourne 1956, gdzie zajął czternaste miejsce wadze półśredniej.

Wicemistrz igrzysk panamerykańskich w 1959 roku.

Letnie Igrzyska Olimpijskie 1956

Przypisy 

Kanadyjscy olimpijczycy
Kanadyjscy zapaśnicy
Uczestnicy Letnich Igrzysk Olimpijskich 1956
Medaliści Igrzysk Panamerykańskich 1959
Urodzeni w 1929
Zmarli w 1990


ZADANIE #2 PRZYGOTOWANIE KOMPONENTÓW - BI-ENCODERA, CROSS-ENCODERA i BM25

In [7]:
!pip install stop-words xformers

In [8]:
#inicjalizacja Bi-Encodera, dokładniej enkodera używanego w obu gałęziach, aby zbudować embeddingi dla tekstów i zapytania
#sam wybierz dokładny model pod kątem j.polskiego
bi_encoder = SentenceTransformer(
    "sdadas/stella-pl-retrieval-mini-8k",
    trust_remote_code=True,
    device="cuda",
    model_kwargs={"torch_dtype":"float16"}
)
query_prefix = "Instruct: Given a web search query, retrieve relevant passages that answer the query.\nQuery: "


#inicjalizacja Cross-encoder do  re-rankowania wyników
#sam wybierz dokładny model pod kątem j.polskiego
cross_encoder = CrossEncoder(
    "sdadas/polish-reranker-roberta-v3",
    activation_fn=torch.nn.Identity(),
    max_length=8192,
    device="cuda",
    model_kwargs={"dtype": torch.float16}
)

# lexical search (keyword search) z użyciem BM25 metody

from rank_bm25 import BM25Okapi
import string
from tqdm import tqdm
from stop_words import get_stop_words

stop_words = get_stop_words('polish')
# Tokenizacja pod BM25 z uwzglednieniem lower case,  usuwaniem stop-words
def bm25_tokenizer(text: str):
    return [word for word in text.lower().split() if word not in stop_words]

tokenized_corpus = []
for passage in tqdm(passages):
    tokenized_corpus.append(bm25_tokenizer(passage))
print("Initialising BM25...")
bm25 = BM25Okapi(tokenized_corpus)
print("Encoding passages...")
corpus_embeddings = bi_encoder.encode(passages, convert_to_tensor=True, show_progress_bar=True)

Loading weights:   0%|          | 0/268 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

100%|██████████| 499874/499874 [01:39<00:00, 5017.34it/s]


Initialising BM25...
Encoding passages...


Batches:   0%|          | 0/15622 [00:00<?, ?it/s]

ZADANIE #3 - FINALNE ROZWIAZANIE - napisz funkcje, która będzie używała BM25 i BI_ENCODERA do wyszukiwania, potem wyniki z obu tych modułów wyszukiwania należy zmergować i dokonać ich re-rankingu i zwrócić top 10 wyników.

In [9]:
# Funkcja, która dla tekstowego zapytania wykona Hybrid Search (BM25, BI-ENCODER), potem dokona RE-RANKINGU i zwróci top 10
def search(query, top_n=10):
    global passages, query_prefix, corpus_embeddings, bi_encoder
    ##### BM25 search (lexical search) #####
    tokenized_query = query.lower().split()
    #policz dopasowanie leksykalne zapytania do każdego dokumentu w korpusie
    doc_scores = bm25.get_scores(tokenized_query)
    #znajdz top N najbliższych dokumentów do zadanego zapytania
    top_bm25 = bm25.get_top_n(tokenized_query, passages, n=top_n)

    ##### Semantic Search using BI-ENCODER #####
    # Encode the query using the bi-encoder and find potentially relevant passages
    query_embedding = bi_encoder.encode(query_prefix+query, convert_to_tensor=True)
    # We use cosine-similarity and torch.topk to find the highest 2 scores
    similarity_scores = bi_encoder.similarity(query_embedding, corpus_embeddings)[0]
    scores, indices = torch.topk(similarity_scores, k=top_n)
    top_bi = [passages[idx] for idx in indices]

    #### MERGE RESULTS FROM BM25 and BI-ENCODER####
    answers = top_bm25 + top_bi

    ##### Re-Ranking #####
    # Now, score all retrieved passages with the cross_encoder
    pairs = [[query, answer] for answer in answers]
    results = model.predict(pairs)

    # RETRIEVE TOP N
    results = [{"answer": pair[1], "score": result, "source": source} for pair, result, source in zip(pairs, results, ["BM25"]*top_n+["BI"]*top_n)]
    results = sorted(results, key=lambda x: x['score'], reverse=True)
    return results


In [10]:
search(query = "Ile mieszkańców ma Warszawa?")

[{'answer': 'Warsaw – miasto w hrabstwie Richmond, stan Wirginia, w Stanach Zjednoczonych. Zgodnie ze spisem ludności z roku 2000 populacja miasta wyniosła 1375. Jest siedzibą hrabstwa Richmond.\n\nGeografia \nWedług Biura Spisów Ludności USA, całkowity obszar miasta to 3 mile kwadratowe (7,9 km²), wszystko z tego jest lądem.\n\nDemografia \nWedług spisu ludności w roku 2000, było tam 1,375 ludzi, 445 gospodarstw domowych i 233 rodziny rezydujące w mieście.',
  'score': np.float32(0.9997111),
  'source': 'BI'},
 {'answer': 'Stoczek – wieś w Polsce położona w województwie podlaskim, w powiecie hajnowskim, w gminie Narewka.\n\nWedług Pierwszego Powszechnego Spisu Ludności z 1921 roku wieś liczyła 20 domów i 111 mieszkańców (67 kobiet i 44 mężczyzn). Większość mieszkańców miejscowości, w liczbie 80 osób, zadeklarowała wyznanie prawosławne, pozostali zgłosili wyznanie rzymskokatolickie (w liczbie 31 osób). Podział religijny mieszkańców wsi całkowicie odzwierciedlał ich strukturą narodowo-e

In [11]:
search(query = "Jak długa jest Wisła?")

[{'answer': 'Vistula Valles (Dolina Wisły) – dolina na powierzchni Marsa o długości 190 km, 23,5° szerokości północnej i na 102,5° długości zachodniej. Nadana w 1985 nazwa nawiązuje do łacińskiej nazwy Wisły.\n\nLinki zewnętrzne \n\n położenie na mapie topograficznej Marsa\n\nTopografia Marsa\nObiekty astronomiczne z nazwami związanymi z Polską',
  'score': np.float32(0.9014003),
  'source': 'BI'},
 {'answer': '(16689) Vistula – planetoida z grupy pasa głównego asteroid okrążająca Słońce w ciągu 5 lat i 223 dni w średniej odległości 3,16 au. Została odkryta 12 sierpnia 1994 roku w Europejskim Obserwatorium Południowym przez Erica Elsta. Nazwa planetoidy pochodzi od łacińskiej nazwy polskiej rzeki Wisły.\n\nZobacz też \n lista planetoid 16001–17000\n lista ponumerowanych planetoid\n\nLinki zewnętrzne \n \n\nPlanetoidy pasa głównego\nPlanetoidy z nazwami związanymi z Polską\nObiekty astronomiczne',
  'score': np.float32(0.89527637),
  'source': 'BI'},
 {'answer': 'Wiślana Trasa Rowerowa 

In [12]:
search(query = "Podaj pięciu polskich Noblistów?")

[{'answer': '1971  \n Prof. Kazimierz Secomski (Polska)\n 1976\n Prof. Walter Isard (USA)\n Prof. Leo Klaassen (Holandia)\n Prof. Józef Górski (Polska)\n 1977\n Prof. Stanisław Leszczycki (Polska)\n 1978\n Prof. Janos Kornai (Węgry)\n 1981\n Prof. Florian Barciński (Polska)\n 1986\n Prof. Zbigniew Zakrzewski (Polska)\n Prof. Walter Klitzsch (Niemcy)\n Prof. Claude Ponsard (Francja)\n Prof. Maciej Wiewiórowski (Polska)\n 1993\n Prof. Stanisław Rączkowski (Polska)\n Prof. Klaus Standke (Niemcy)\n Prof. Cees Veeger',
  'score': np.float32(0.88323104),
  'source': 'BI'},
 {'answer': '1964\n Prof. Kamil Stefko \n 1965\n Prof. Michał Kalecki \n 1972 \n Prof. Henryk Jabłoński \n Prof. Artur Bordag \n Prof. Stefan Górniak \n 1977\n Prof. Antoni Wrzosek \n 1978 \n Prof. Tigran Siergiejewicz Chaczaturow \n 1986 \n Prof. Jochen Schumann \n 1988\n Prof. Jurij Aleksandrowicz Ławrikow\n 1990 \n Prof. Aleksander Grigorjewicz Granberg\n 1991\n Prof. Giuseppe Calzoni \n 1992\n Prof. Robert Leroy King \

In [13]:
search(query = "Jak nazywał się pierwszy król Polski?")

[{'answer': 'Polskie królowe – wykaz ten obejmuje koronowane lub używające tytułu królewskiego żony polskich władców. Wykaz koronacji polskich królowych (i królów) zobacz: Lista koronacji królów i królowych polskich\n\nPierwszym koronowanym królem Polski był Bolesław I Chrobry, jednak nie wiadomo, czy jego żona - Oda, margrabianka miśnieńska, dożyła koronacji męża w 1025 roku. Pierwszą pewną polską królową jest Rycheza, żona Mieszka II.\n\nW 1076 roku na króla Polski',
  'score': np.float32(0.99097323),
  'source': 'BI'},
 {'answer': 'Turniej o Koronę Bolesława Chrobrego – Pierwszego Króla Polski – towarzyski turniej żużlowy, rozgrywany w Gnieźnie. Pierwszy turniej odbył się 29 czerwca 2008 roku. Był to pierwszy od ośmiu lat indywidualny turniej żużlowy rozegrany w Gnieźnie. Organizatorzy obsadzili turniej zawodnikami z Ekstraligi, pierwszej ligi oraz zawodnikami miejscowymi. Poprzedni towarzyski turniej indywidualny - Speedway Gniezno 2000 - odbył się 30 września 2000 roku, a wygrał',